# EDA workflow template — from raw data to a validated dataset

Orchestrates the four skills as one leakage-safe pipeline. **Scope:** EDA and
dataset quality for a tree / neural / generative model. Model training and
hyper-parameter tuning are **out of scope**; simple models / AE·VAE / pretrained
encoders appear only as diagnostic probes.

Discipline enforced throughout:
- keep an immutable raw layer;
- **split before** any data-dependent `fit`;
- fit imputers/scalers/encoders/PCA/selectors/samplers **only on train/fold**;
- keep validation/test at the natural deployment prevalence;
- label each claim as fact / interpretation / hypothesis.

Replace the synthetic-data cell with your own load to reuse the template.

In [ ]:
import sys, pathlib
import numpy as np, pandas as pd

# Make the four skills' scripts importable (this notebook lives in
# plan-eda-dataset/assets/, so the skills root is two levels up).
ROOT = pathlib.Path.cwd()
while not (ROOT / "plan-eda-dataset").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ("audit-eda-data-quality", "discover-eda-structure",
            "engineer-select-eda-features", "plan-eda-dataset"):
    sys.path.insert(0, str(ROOT / sub / "scripts"))

import contracts as C
pd.set_option("display.width", 120)

## Stage 0 — frame, protect evaluation, split (`plan-eda-dataset`)

Define the dataset contract, snapshot the immutable raw layer, and choose a split
whose mechanism matches how new data appear. Nothing data-dependent is fitted yet.

In [ ]:
from sklearn.datasets import make_classification
import split_designer as sd

# --- replace this block with your own data load -------------------------------
X, y = make_classification(n_samples=2000, n_features=10, n_informative=5,
                           n_redundant=3, weights=[0.95, 0.05], class_sep=1.1,
                           random_state=42)
cols = [f"f{i}" for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=cols)
df["churn"] = y
rng = np.random.default_rng(42)
df["tags"] = rng.choice(["a", "b", "(a, b)", "(b, c)", "c", "(a, c, d)"], size=len(df))
df.loc[rng.choice(len(df), 150, replace=False), "f0"] = np.nan
# ------------------------------------------------------------------------------

RAW = df.copy(deep=True)          # immutable raw layer — never mutated in place
TARGET = "churn"

contract = C.DatasetContract(
    unit_of_observation="customer", intended_decision="retention outreach",
    target=TARGET, prediction_horizon="30d", keys=[], time_column=None,
    readiness_criteria=["no split/target/temporal leakage", "stable selection"],
)

split = sd.make_split(df, strategy="stratified_random", target=TARGET, test_size=0.2)
train = df.iloc[split["train_idx"]].reset_index(drop=True)
test = df.iloc[split["test_idx"]].reset_index(drop=True)
print("split manifest:", split["manifest"]["overlap_checks"],
      "| rows:", split["manifest"]["row_counts"])
print("train churn rate:", round(train[TARGET].mean(), 4),
      "| test churn rate:", round(test[TARGET].mean(), 4))

## Stage 1 — audit quality & distributions, diagnose imbalance (`audit-eda-data-quality`)

Measure before repairing. Fit anomaly detection on train only. Diagnose the
class imbalance here; the resampling decision belongs to Stage 3.

In [ ]:
import profile_schema as ps, distribution_report as dr, missingness as ms
import outliers as ol, leakage_checks as lc

schema = ps.profile_schema(train, target=TARGET)
display(schema[["role", "dtype", "missing_rate", "n_unique", "issues"]])

display(dr.numeric_summary(train, cols).round(3))
display(ms.missingness_vs_target(train, TARGET).head())

# multivariate outliers: fit on train, score test (no refit on test)
out_test = ol.multivariate_outliers(train, cols, score_df=test)
print("flagged outliers in test:", int(out_test["is_outlier"].sum()), "/", len(test))

# leakage screen + imbalance diagnosis (diagnose only, do not resample here)
susp = lc.suspicious_single_features(train[cols], train[TARGET])
display(susp.head())
print("positives — train:", int(train[TARGET].sum()), "| test:", int(test[TARGET].sum()))

## Stage 2 — discover structure (`discover-eda-structure`)

Associations and redundancy, cluster tendency before clustering, and a train-only
PCA. "No stable clusters" is a valid finding.

In [ ]:
import associations as asc, clustered_correlation as cc
import cluster_tendency as ct, clustering as cl, dim_reduction as dm

print("assoc f0~f1:", asc.auto_association(train, "f0", "f1"))

blocks = cc.redundancy_blocks(train[cols], abs_threshold=0.8)
print("redundancy blocks:", blocks["blocks"])

print("Hopkins (>0.5 hints at clusterability):",
      round(ct.hopkins_statistic(train[cols].fillna(0)), 3))
km = cl.run_clustering(train[cols].fillna(0), algorithm="kmeans", k=3)
print("internal indices:", cl.internal_indices(km["X"], km["labels"]))
print("stability (mean ARI):",
      cl.cluster_stability(train[cols].fillna(0), k=3, n_boot=10)["mean_ari"])

pca = dm.fit_reduce(train[cols].fillna(0), method="pca", n_components=3)
print("PCA explained variance (train-fit):", pca["explained_variance"])
_ = dm.apply_reduce(pca, test[cols].fillna(0))  # transform test with train fit

## Stage 3 — engineer, select, balance (`engineer-select-eda-features`)

Build features (train-fit), select with filter + mRMR + wrapper evidence, then
decide balancing. For 5% positives with enough absolute count, prefer class
weights + threshold tuning over hard resampling.

In [ ]:
import feature_builders as fb, filter_select as fs, mrmr
import wrapper_embedded as we, balancing as bl
from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier

# CDF feature: fit ECDF on train, apply to test (no refit)
cdf = fb.EmpiricalCDF().fit(train[["f1"]].to_numpy())
train["f1_cdf"] = cdf.transform(train[["f1"]].to_numpy())
test["f1_cdf"] = cdf.transform(test[["f1"]].to_numpy())

# mixed-category multi-hot with a train vocabulary
enc = fb.MixedCategoryMultiHot(name="tags").fit(train["tags"])
tags_tr = enc.transform(train["tags"])
print("multi-hot columns:", list(tags_tr.columns))

feat_cols = cols + ["f1_cdf"]
rel = fs.relevance_scores(train[feat_cols].fillna(0), train[TARGET])
sel = mrmr.mrmr_select(train[feat_cols].fillna(0), train[TARGET], k=6)
print("mRMR selected:", sel["selected"])

light = DecisionTreeClassifier(max_depth=5, random_state=0)
sig = we.paired_feature_significance(train[feat_cols].fillna(0), train[TARGET],
                                     features_to_test=["f1_cdf"], estimator=light,
                                     mode="cv", n_runs=5, cv=5, scoring="f1")
print("paired A/B for f1_cdf:", sig)

# balancing DECISION: weights first; moderate resample only if it helps (train only)
print("class weights:", bl.compute_class_weights(train[TARGET]))
Xr, yr, samp = bl.random_resample(train[feat_cols].fillna(0), train[TARGET],
                                  kind="over", target_ratio=0.2)
print("resample manifest:", {k: samp[k] for k in ("algorithm", "original_counts",
                                                   "resampled_counts")})

## Readiness gate (`plan-eda-dataset`)

Re-check the produced dataset and emit the handoff manifests + verdict. A failing
gate returns `not_ready` with the smallest corrective experiment.

In [ ]:
import readiness_check as rc

train_out = train.assign(_id=np.arange(len(train)))
test_out = test.assign(_id=np.arange(len(train), len(train) + len(test)))
checks = rc.run_structural_checks(train_out, test_out, key_cols=["_id"])
verdict = rc.readiness_gate(
    checks,
    corrective_experiment="if not_ready: re-run the stage that fixes the named check",
)
print("READINESS:", verdict["verdict"])
for k, (ok, ev) in checks.items():
    print(f"  [{'x' if ok else ' '}] {k}: {ev}")

manifest_path = str(ROOT / "eda_handoff.json")
C.save_manifests(
    manifest_path,
    dataset_contract=contract,
    split=split["manifest"],
    sampling=samp,
    selection={"mrmr_selected": sel["selected"], "paired_ab_f1_cdf": sig},
    readiness=verdict,
)
print("saved:", manifest_path)

### Handoff & forbidden conclusions

- Deliver: dataset card, feature/transformation/split/sampling manifests, selection
  and diagnostic reports, unresolved risks.
- Do **not** present SHAP / importance / correlation / MI / clusters as causal.
- Do **not** report a test metric as an EDA result; the test set stays untouched.

## Appendix — time-series data (optional)

If your data is a time series (a time column, possibly several entities), the
same discipline applies with time-specific steps: put each entity on a **regular
grid**, check **stationarity** as a *diagnostic*, build **leakage-safe lags /
rolling** with `shift`, and split **chronologically** (test strictly later).

In [ ]:
import time_series_features as tsf
from split_designer import make_split

# --- synthetic panel series: 2 machines, hourly, trend + daily season + noise ---
rng = np.random.default_rng(0)
frames = []
for m in ["m1", "m2"]:
    t = pd.date_range("2024-01-01", periods=500, freq="h")
    trend = np.linspace(0, 5, 500)
    season = 3 * np.sin(2 * np.pi * np.arange(500) / 24)
    noise = np.cumsum(rng.standard_normal(500)) * 0.3
    frames.append(pd.DataFrame({"machine": m, "ts": t, "sensor": 20 + trend + season + noise}))
ts_df = pd.concat(frames, ignore_index=True)
ts_df = ts_df.drop(index=rng.choice(len(ts_df), 30, replace=False)).reset_index(drop=True)  # gaps

# 1) regular grid per machine, filled ONLY from the past
ts_df = tsf.to_regular_grid(ts_df, "ts", freq="h", group="machine")

# 2) stationarity DIAGNOSTIC (ADF/KPSS if statsmodels present, else rolling stats)
st = tsf.stationarity_report(ts_df.query("machine == 'm1'")["sensor"])
print("stationarity(m1):", st.get("adf") or st["rolling"])

# 3) leakage-safe features within each machine (shift before rolling; seasonal diff)
ts_df = tsf.add_lags(ts_df, "sensor", lags=[1, 24], group="machine")
ts_df = tsf.add_rolling(ts_df, "sensor", windows=[24], funcs=["mean", "std"], group="machine", shift=1)
ts_df = tsf.add_difference(ts_df, "sensor", periods=[1], seasonal=24, group="machine")

# 4) chronological split (test is strictly later in time)
ready = ts_df.dropna().reset_index(drop=True)
sp = make_split(ready, strategy="chronological", time_col="ts", test_size=0.2)
print("split boundaries:", sp["manifest"]["time_boundaries"])
print("rows:", sp["manifest"]["row_counts"],
      "| time_ordered:", sp["manifest"]["overlap_checks"].get("time_ordered"))